# 07 · Experiment E2: which loss? (task T23, runs R10 to R15)

**RQ2.** Water is under 10% of the labelled pixels. Which loss copes best with that imbalance? Everything else is the frozen default (MobileNetV2 encoder, VV + VH + ratio input):

| Config | Loss | Runs |
|---|---|---|
| BCE loss | binary cross-entropy, averaged over valid pixels | R10, R11, R12 |
| Dice loss | 1 − Dice overlap with the water class | R13, R14, R15 |
| Default | BCE + Dice | R01, R02, R03 (shared with E1) |

BCE judges every pixel on its own, so the many easy "not water" pixels dominate it; Dice scores the overlap with the water class directly. Precision and recall are reported next to IoU because the losses are expected to trade one for the other.

In [ ]:
import json
import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "config.py").exists())
sys.path.insert(0, str(ROOT))
import src  # noqa: F401  (must come before TensorFlow: switches on the XLA compile cache)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src import analysis, viz
from src.config import STUDIES, load_config, run_table, runs_of
from src.runner import check_frozen, run_benchmark, run_evaluation, run_many

viz.use_style()
pd.set_option("display.float_format", lambda v: f"{v:.4f}")
print("training and scoring code is identical to the frozen tag 'defaults-frozen' (commit %s)" % check_frozen()[:7])

STUDY = "E2"
CONFIGS = STUDIES[STUDY]["configs"]
pd.DataFrame([r for r in run_table() if r["config_name"] in CONFIGS]).fillna({"loss": "bce_dice"}).set_index("run_id")

## 1. Train and score R10 to R15

In [ ]:
results = run_many(["R10", "R11", "R12", "R13", "R14", "R15"])

## 2. Results

One row per run. Validation IoU is the score the checkpoint was chosen by; test and Bolivia were scored once, after training.

In [ ]:
runs = analysis.runs_table()
study_runs = runs[runs["Config"].isin(CONFIGS)]
study_runs[["Run ID", "Config", "Seed", "Status", "Epochs trained", "Best epoch", "Val IoU (best)", "Test IoU",
            "Test precision", "Test recall", "Bolivia IoU", "Sec per epoch"]].set_index("Run ID")

Mean ± standard deviation over the three seeds (the two threshold baselines, single deterministic runs, are shown for reference):

In [ ]:
analysis.study_table(STUDY, analysis.summary_table(runs))

**Is a difference real?** The project's rule: a gap between two configs counts only if it is larger than the seed-to-seed standard deviation.

In [ ]:
gaps = analysis.gap_table(STUDY, runs)
gaps.to_csv(ROOT / "results" / f"{STUDY}_gaps.csv", index=False)
gaps.set_index(["Metric", "A", "B"])

In [ ]:
hist = {c: analysis.histories(runs_of(c)) for c in CONFIGS}
axes = viz.plot_val_curves(hist)
axes[0].figure.tight_layout()
viz.savefig(axes[0].figure, ROOT / "figures" / f"{STUDY}_val_curves.png")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.3), sharey=True)
done = runs[runs["Status"] == "Finished"]
for ax, metric in zip(axes, ("Test IoU", "Bolivia IoU")):
    viz.plot_config_bars(done, CONFIGS, metric, ax, baseline=("Otsu (E0)", "Tuned VH threshold"))
    ax.set_title(metric.replace("IoU", "water IoU"))
fig.tight_layout()
viz.savefig(fig, ROOT / "figures" / f"{STUDY}_bars.png")
plt.show()

Precision and recall on the test split, per run:

In [ ]:
pr = study_runs[["Run ID", "Config", "Seed", "Test precision", "Test recall", "Test F1", "Bolivia precision", "Bolivia recall"]]
pr.set_index("Run ID")